# 18 - Live API
Run the setup cell first, then the work cells in order. Use the WSL mlop-pj kernel.
These cells change cloud resources when you run them. A pending result needs a wait and a phase retry, not a new job.
Use notebooks 18-22 for the five main parts. Training and log waits can overlap with GitHub setup. See docs/RUN_FINISH.md.


In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert Path(sys.prefix).name == "mlop-pj", "Select the WSL mlop-pj kernel."
assert "microsoft" in os.uname().release.lower(), "Open this project in WSL."
status = json.loads((ROOT / "reports/azure-blob-upload.json").read_text())
assert status.get("data_uploaded") and status.get("all_hashes_match") and not status["errors"], "Finish 17 first."
from cloudlayer.finish import run

def step(phase):
    print("Running:", phase, flush=True)
    result = run(phase)
    print(json.dumps(result, indent=2))
    if result["errors"]:
        raise RuntimeError(f"Check reports/finish-{phase}.json and the matching log in tmp. Stop this group.")
    if result.get("pending"):
        print("Still waiting. Read next in the report; do not recreate the resource or job.")
    return result

print("Root:", ROOT)
print("17 passed. Run this notebook work cells in order.")

Root: /mnt/c/Users/SN23341J001185/Documents/MyWork/ICT/ICTYear4_1/MLop/public_teaching_mlaiops/course/MLOP-PJ
17 passed. Run this notebook work cells in order.


## Package setup
Install the Azure DVC plugin once. Constraints keep the existing model package versions. This installs packages; it does not train or deploy. If installation fails, stop and send the last error.

In [2]:
import importlib.util
if importlib.util.find_spec("dvc_azure") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-cloud.txt",
                    "-c", "requirements-notebook.lock.txt", "-c", "requirements-local.lock.txt"],
                   cwd=ROOT, check=True)
print("Azure DVC plugin ready.")

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/56/8c/2a895811f69699915ddd4e346ea3a1ab26b585025eac890d19dd5193c7f4/adlfs-2026.8.0-py3-none-any.whl (46 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/46/bd/551ee6af426af84ca33e02622be722925c196608e9127d731ef17c47f06e/argcomplete-3.7.2-py3-none-any.whl (43 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/5b/db/325c6d7312d2200251c52323878281045aaffcb5586612296484e4280eaa/azure_core-1.41.0-py3-none-any.whl (220 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/b2/d5/ae45ce8009d0d015304b206c7c4a6cbab02c03f8f07eedefb32b099ccee3/azure_identity-1.26.0-py3-none-any.whl (195 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/9b/57/d1f45fbccc0dfbe6b6db7e5fa06199e35219c712f3743677bec1b4e7d78b/azure_storage_blob-12.31.0-py3-none-any.whl (455 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/88/5a/e78383666d3814d095c1de


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Group 1 — DVC, image, and live API
`assets` pushes the raw-data DVC cache and checks a fresh pull against the raw SHA-256. It tests the code, builds the updated API image, and pushes it to ACR. `deploy` creates the API pull identity and Container App, then checks readiness, golden cases, invalid input, batch input, and warm load.

Inputs: the completed Blob upload, canonical bundle, source files, and pinned image base. Outputs: `finish-assets.json`, `finish-deploy.json`, `azure-smoke.json`, and `azure-load.json` in reports. Look for `assets_ready: true` and `api_ready: true`. Open the returned URL with `/docs` to try the API. The warm p95 target is 300 ms at concurrency 10; `api_ready` alone does not prove that target passed.

This group can take several minutes. If a command times out, check existing resources before retrying.

In [3]:
assets = step("assets")
assert assets.get("assets_ready"), "Finish the asset checks first."
api = step("deploy")
assert api.get("api_ready"), "Fix the API checks before continuing."

Running: assets
{
  "checked_at": "2026-10-11T16:26:31.745143+00:00",
  "phase": "assets",
  "errors": [],
  "local_tests_passed": true,
  "dvc_fresh_pull_passed": true,
  "model_version": "abfd88e893ca",
  "artifact_sha256": "71f3ba41b4b964f5beb31e3d6e13b835b1dd2538aa663741d8837301be9dff13",
  "source_sha256": "77ea7d031dd98b7c1fcf66be162c3e1f6508dad7a6432b192dc35a85d46aedeb",
  "local_image": "mlop-pj-api:finish-abfd88e893ca-77ea7d031dd9",
  "image_uri": "acrmloppj492ea24d.azurecr.io/travel-style@sha256:05d1819321f472b39fe1035cb9cee68fc7432a9ad2be4b56afdc8c85af562fcc",
  "assets_ready": true
}
Running: deploy
{
  "checked_at": "2026-10-11T16:27:40.865079+00:00",
  "phase": "deploy",
  "errors": [
    "RuntimeError: Azure CLI failed: ExpressEnvironmentFeatureNotSupported"
  ],
  "cloud_writes_started": true
}


RuntimeError: Check reports/finish-deploy.json and the matching log in tmp. Stop this group.

## Cold-start measurement
Run after the warm test. Close `/docs` and stop other requests first. This cell checks whether the app has zero replicas before timing a new request. If replicas do not reach zero yet, the report stays pending. Keep cold latency separate from warm p95.

In [ ]:
cold = step("cold")